# Charmonia CNM Components: LHC p+Pb 5.02 & 8.16 TeV

This notebook analyzes the individual contribution of **Energy Loss** and **pT Broadening** components to the nuclear modification factor $R_{pPb}$ for Charmonia (J/psi) at LHC energies.

It uses the `CNMCombine` class to load physics parameters and `eloss_cronin_centrality` for visualization.

In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add parent directory to path to import modules
sys.path.append("..")

from cnm_combine.cnm_combine import CNMCombine, DEFAULT_CENT_BINS
from eloss_code.eloss_cronin_centrality import (
    plot_RpA_vs_y_components_per_centrality,
    plot_RpA_vs_pT_components_per_centrality,
    plot_RpA_vs_centrality_components_band,
    set_publication_style
)

# Apply generic publication style
set_publication_style()

ENERGIES = ["5.02", "8.16"]
OUTDIR = Path("../outputs/LHC/CNM_Components")
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Output Directory: {OUTDIR}")

Output Directory: ../outputs/LHC/CNM_Components


## Components Analysis Loops

We iterate over both LHC energies to generate component plots.

In [2]:
for energy in ENERGIES:
    print(f"\n=== Analyzing p+Pb {energy} TeV ===")
    
    # 1. Initialize CNM System
    cnm = CNMCombine.from_defaults(energy)
    
    P = cnm.P
    roots = cnm.roots
    qp = cnm.qp
    gl = cnm.gl
    
    # Note: Use default CMS-like y-windows from the class
    y_wins = cnm.y_windows
    cent_bins = cnm.cent_bins
    
    # --------------------------------------------------------
    # A. Components vs Rapidity (y)
    # --------------------------------------------------------
    y_axis = np.linspace(-5.0, 5.0, 101)
    pT_avg = 0.0
    
    # We iterate manually to call the plotting function
    fig_y, axes_y = plot_RpA_vs_y_components_per_centrality(
        P, roots, qp, gl,
        cent_bins,
        y_axis, pT_avg,
        show_components=("loss", "broad", "total", "npdf", "cnm"),
        q0_pair=cnm.q0_pair,
        p0_scale_pair=(0.9, 1.1),
        ncols=3,
        suptitle=rf"$J/\psi$ p+Pb {energy} TeV: Components vs $y$"
    )
    plt.savefig(OUTDIR / f"RpA_components_vs_y_pPb{energy.replace('.', 'p')}.pdf", bbox_inches="tight")
    plt.show()
    
    # --------------------------------------------------------
    # B. Components vs pT
    # --------------------------------------------------------
    pT_edges = np.linspace(0, 15, 31)
    
    for y0, y1, label in y_wins:
        fig_pt, axes_pt = plot_RpA_vs_pT_components_per_centrality(
            P, roots, qp, gl,
            cent_bins,
            pT_edges, (y0, y1),
            show_components=("loss", "broad", "total", "npdf", "cnm"),
            q0_pair=cnm.q0_pair,
            p0_scale_pair=(0.9, 1.1),
            ncols=5, # 5 bins (0-20, 20-40, 40-60, 60-80, 80-100)
            step=True,
            suptitle=rf"$J/\psi$ p+Pb {energy} TeV ({label}): Components vs $p_T$"
        )
        # Clean filename
        safe_label = label.replace(" ", "_").replace("<", "lt").replace(">", "gt")
        plt.savefig(OUTDIR / f"RpA_components_vs_pT_pPb{energy.replace('.', 'p')}_{safe_label}.pdf", bbox_inches="tight")
        plt.show()

    # --------------------------------------------------------
    # C. Components vs Centrality
    # --------------------------------------------------------
    pT_integ = (0.0, 30.0) # Integrate over all pT
    
    # Create a figure with subplots for each rapidity window
    fig_c, axes_c = plt.subplots(1, len(y_wins), figsize=(5 * len(y_wins), 5), sharey=True)
    if len(y_wins) == 1:
        axes_c = [axes_c]
        
    for i, (y0, y1, label) in enumerate(y_wins):
        ax = axes_c[i]
        
        # Calculate bands explicitly or allow the plotter to do it
        # We use the CNMCombine method to get consistent data
        (labels, RL_c, RL_lo, RL_hi, RB_c, RB_lo, RB_hi, 
         RT_c, RT_lo, RT_hi, 
         RMB_loss, RMB_broad, RMB_tot) = cnm._calc_eloss_broad_band_vs_centrality(
             cent_bins, (y0, y1), pT_integ
         )

        plot_RpA_vs_centrality_components_band(
            cent_bins, labels,
            RL_c, RL_lo, RL_hi, RMB_loss,
            RB_c, RB_lo, RB_hi, RMB_broad,
            RT_c, RT_lo, RT_hi, RMB_tot,
            show=("loss", "broad", "total", "npdf", "cnm"),
            ax=ax,
            ylabel=rf"$R_{{pPb}}$" if i == 0 else "",
            note=f"{label}\n{energy} TeV",
            system_label=""
        )
    
    fig_c.tight_layout()
    plt.savefig(OUTDIR / f"RpA_components_vs_cent_pPb{energy.replace('.', 'p')}.pdf", bbox_inches="tight")
    plt.show()


=== Analyzing p+Pb 5.02 TeV ===
[OpticalGlauber] Target A=208, d=0.549 fm, σ_nn=67.00 mb
[OpticalGlauber] ∫ d²s T_A(s) ≈ 208.483  (should be ~A=208)
[OpticalGlauber] ∫ d²s T_d(s) ≈ 1.9972  (should be ~2)
[OpticalGlauber] Tabulating overlaps T_AA(b), T_pA(b), T_dA(b)...
[OpticalGlauber] σ_tot: AA=7718.4 mb, pA=1909.0 mb, dA=2361.5 mb


NameError: name 'F2_t' is not defined